# 1. Import of input data

In [ ]:
import nbimporter
import sys
import os
from pathlib import Path

In [ ]:
# Import libraries
try:
    current_dir = RUN_DIR
    print(f"📂 Boiler.ipynb executed by RUN.ipynb — using directory: {current_dir}")
except NameError:
    current_dir = Path().resolve()
    print(f"📁 Boiler.ipynb executed standalone — using directory: {current_dir}")

Libraries_path = (current_dir / "Libraries.ipynb").resolve()

# If not found, try in the parent levels
if not Libraries_path.exists():
    Libraries_path = (current_dir.parent / "Libraries.ipynb").resolve()

# Run the file
if Libraries_path.exists():
    print(f"📘 Importing libraries from: {Libraries_path}")
    get_ipython().run_line_magic('run', f'"{Libraries_path}"')
else:
    print("⚠️ Libraries.ipynb not found! Some functions might not be available.")

In [ ]:
# Database parameter import
energysystem = pd.read_excel(energysystem_path, sheet_name = 'Boiler')

# 2. Input parameters

In [ ]:
# DASHBORD - VALUE INPUTS
Power = energysystem['Thermal capacity [kW]']
Energy_produced = energysystem['Energy produced [kWh /year]']
Efficiency = energysystem['Efficiency (between 0 and 1)']
EnergyInputSource = energysystem['Energy source (Gas-blend, LPG, Biomethane, Hydrogen)']


In [ ]:
# DASHBOARD - INPUT PARAMETERS
Power = Power

Efficiency_LHV = Efficiency

# DEFAULT PARAMETERS
# Natural gas
LHV = 36.8      # MJ/m3
HHV = 40.4      # MJ/m3

# LPG
LHVlpg = 93     # MJ/kg
HHVlpg = 102    # MJ/kg

# Biomethane as grid-gas substitute
LHVBiom = 35    # MJ/m3
HHVBiom = 38    # MJ/m3

# Hydrogen
LHVH2 = 120     # MJ/kg
HHVH2 = 142     # MJ/kg 

Efficiency_HHV = Efficiency_LHV*(LHV/HHV)  # Efficiency based on HHV

TotEnergyNeeds = np.array(Energy_produced)  # Energy need for 1 year kWh per year

#Define lifespan
Lifespan = np.where(Power <= 70, 19, np.where(Power <= 400, 21, 26))

def calculate_mass(Power):
    if Power < 24.5:
        return 7.430 * np.log(Power) + 11.875
    elif 24.5 <= Power <= 100:
        return 140.86 * np.log(Power) - 415
    else:
        i = np.ceil(Power / 100)
        return (140.86 * np.log(Power / i) - 415) * i

mass = Power.apply(calculate_mass)

# 3. Life cycle inventories

## 3.1 Module A1-3

In [ ]:
Boiler_stainlessSteel = 0.156 * mass
Boiler_steel = 0.532 * mass
Boiler_aluminium = 0.128 * mass
Boiler_elastomer = 0.0003 * mass
Boiler_PVC = 0.003 * mass
Boiler_refrigerantgas = 0.065 * mass
Boiler_copper = 0.098 * mass
Boiler_electronic = np.ones(len(Power)) #kg
Boiler_welding = np.where((0 <= Power) & (Power <= 100), 
                          np.full(len(Power), 4),
                          np.where((100 < Power) & (Power <= 1000), 
                                   np.full(len(Power), 6),
                                   np.full(len(Power), 8)
                                  )
                         )

Boiler_waterconsumption = 6.21 * Power + 119.89
# water consumption in kg; emissions and wastewater in m3
Boiler_emission_water_air = 0.15 * (Boiler_waterconsumption / 1000)
Boiler_emission_water_water = 0.02 * (Boiler_waterconsumption / 1000)
Boiler_wastewater_treatment = 0.83 * (Boiler_waterconsumption / 1000)
Boiler_lubricating_oil = 0.27 * Power
Boiler_A13electricity = 0.59 * Power
Boiler_heat = 5.15 * Power
Boiler_hazardous = np.where((0 <= Power) & (Power <= 100), 
                            np.full(len(Power), 0.2),
                            np.where((100 < Power) & (Power <= 1000), 
                                     np.full(len(Power), 0.3),
                                     np.full(len(Power), 0)
                                    )
                           )


## 3.2 Module A4

In [ ]:
Boiler_Truk_16_32 = ((500 + 200)*(mass)/1000)
Boiler_Light_Truk = (50*(mass)/1000)

## 3.3 Module B2

In [ ]:
Boiler_Light_Truk_B2 = 50 * (Lifespan/2) * 0.01 * (mass/1000)

## 3.4 Module B4

In [ ]:
BoilerNumberReplacements = (np.ceil(np.divide(ReqSL, Lifespan) - 1))
SFBoiler = (RSP / ReqSL)  # Scaling factor

## 3.5 Module B6

In [ ]:
GasBlend   = np.where(EnergyInputSource == 'Gas-blend', 1, 0)
Biomethane = np.where(EnergyInputSource == 'Biomethane', 1, 0)
Hydrogen   = np.where(EnergyInputSource == 'Hydrogen', 1, 0)
LPG        = np.where(EnergyInputSource == 'LPG', 1, 0)

In [ ]:
TotEnergyNeeds = TotEnergyNeeds

# ------------------------------------------------------------------
# B6 – fuel balance and ecoinvent inputs
# All fuel energies are expressed on an LHV basis.
# ------------------------------------------------------------------

# Useful heat delivered during the reference study period
boiler_heat_output_kWh = np.asarray(Energy_produced, dtype=float) * RSP
boiler_heat_output_MJ = boiler_heat_output_kWh * 3.6

# Fuel energy required at boiler inlet [MJ fuel, LHV]
boiler_fuel_energy_MJ = boiler_heat_output_MJ / np.asarray(Efficiency_LHV, dtype=float)

# Energy shares by fuel


boiler_natural_gas_share = np.where(
    EnergyInputSource == 'Gas-blend',
    A,
    0.0
)

boiler_biomethane_share = np.where(
    EnergyInputSource == 'Gas-blend',
    B,
    np.where(EnergyInputSource == 'Biomethane', 1.0, 0.0)
)

boiler_hydrogen_share = np.where(
    EnergyInputSource == 'Gas-blend',
    C,
    np.where(EnergyInputSource == 'Hydrogen', 1.0, 0.0)
)

boiler_lpg_share = np.where(
    EnergyInputSource == 'LPG',
    1.0,
    0.0
)

# Fuel energy by carrier [MJ fuel, LHV]
boiler_natural_gas_energy_MJ = boiler_fuel_energy_MJ * boiler_natural_gas_share
boiler_biomethane_energy_MJ = boiler_fuel_energy_MJ * boiler_biomethane_share
boiler_hydrogen_energy_MJ = boiler_fuel_energy_MJ * boiler_hydrogen_share
boiler_lpg_energy_MJ = boiler_fuel_energy_MJ * boiler_lpg_share

# Physical fuel inputs used by the corresponding ecoinvent datasets
boiler_natural_gas_input_m3 = boiler_natural_gas_energy_MJ / LHV
boiler_biomethane_input_m3 = boiler_biomethane_energy_MJ / LHVBiom
boiler_lpg_input_kg = boiler_lpg_energy_MJ / LHVlpg

# Hydrogen mass entering the boiler [kg H2]
boiler_hydrogen_input_kg = (
    boiler_hydrogen_energy_MJ
    / LHVH2
)

# Electricity attributed to H2 production [kWh electricity]
# 55 kWh_el per kg_H2
boiler_hydrogen_electrolysis_electricity_kWh = (
    boiler_hydrogen_input_kg
    * 55
)

# Auxiliary electricity for boiler operation [kWh]
boiler_auxiliary_electricity_kWh = 0.0056 * boiler_fuel_energy_MJ

Boiler_electricity = boiler_auxiliary_electricity_kWh * Grid
Boiler_electricityPVExt = boiler_auxiliary_electricity_kWh * PV

Boiler_electricityRECSPV = Boiler_electricity * RECSPV
Boiler_electricityRECSHydro = Boiler_electricity * RECSHydro
Boiler_electricityRECSEolic = Boiler_electricity * RECSEolic

# ------------------------------------------------------------------
# B6 – direct emissions from fuel combustion
# Emission factors are expressed per MJ of fuel input, LHV basis.
# ------------------------------------------------------------------

# Non-H2 fuel energy: used for pollutants generated by combustion
boiler_combustion_energy_MJ = (
    boiler_natural_gas_energy_MJ
    + boiler_biomethane_energy_MJ
    + boiler_lpg_energy_MJ
)

# CO2 is distinguished by carbon origin
boiler_fossil_combustion_energy_MJ = (
    boiler_natural_gas_energy_MJ
    + boiler_lpg_energy_MJ
)

boiler_biogenic_combustion_energy_MJ = boiler_biomethane_energy_MJ

Boiler_Acetaldehyde = 1.00E-09 * boiler_combustion_energy_MJ
Boiler_AceticAcid = 1.50E-07 * boiler_combustion_energy_MJ
Boiler_Benzene = 4.00E-07 * boiler_combustion_energy_MJ
Boiler_Benzoapyrene = 1.00E-11 * boiler_combustion_energy_MJ
Boiler_Butane = 7.00E-07 * boiler_combustion_energy_MJ

# Natural gas 0.056 kg/MJ (LHV); LPG 0.0631 kg/MJ (LHV) - IPCC 2006 Guidelines, Vol. 2, Table 2.2
Boiler_CarbonDioxideFossil = (
    5.60E-02 * boiler_natural_gas_energy_MJ
    + 6.31E-02 * boiler_lpg_energy_MJ
)

Boiler_CarbonDioxideBiogenic = (
    5.60E-02 * boiler_biogenic_combustion_energy_MJ
)

Boiler_CarbonMonoxideFossil = 3.00E-05 * boiler_combustion_energy_MJ
Boiler_DinitrogenMonoxide = 5.00E-07 * boiler_combustion_energy_MJ
Boiler_Formaldehyde = 1.00E-07 * boiler_combustion_energy_MJ
Boiler_Mercury = 3.00E-11 * boiler_combustion_energy_MJ
Boiler_MethaneFossil = 2.00E-06 * boiler_combustion_energy_MJ
Boiler_NitrogenOxides = 2.00E-05 * boiler_combustion_energy_MJ
Boiler_PAH = 1.00E-08 * boiler_combustion_energy_MJ
Boiler_Particulates = 1.00E-07 * boiler_combustion_energy_MJ
Boiler_Pentane = 1.20E-06 * boiler_combustion_energy_MJ
Boiler_Propane = 2.00E-07 * boiler_combustion_energy_MJ
Boiler_PropionicAcid = 2.00E-08 * boiler_combustion_energy_MJ
Boiler_SulfurDioxide = 5.00E-07 * boiler_combustion_energy_MJ
Boiler_Toluene = 2.00E-07 * boiler_combustion_energy_MJ
Boiler_Nitrate = 1.30E-07 * boiler_combustion_energy_MJ
Boiler_Nitrite = 3.00E-09 * boiler_combustion_energy_MJ
Boiler_Sulfate = 5.00E-08 * boiler_combustion_energy_MJ
Boiler_Sulfite = 5.00E-08 * boiler_combustion_energy_MJ

## 3.6 Module C2

In [ ]:
Boiler_Transport_to_treatment_plant = 50 * (mass/1000)

## 3.7 Module C3

In [ ]:
Boiler_EOL_stainlessSteel = Boiler_stainlessSteel
Boiler_EOL_steel = Boiler_steel
Boiler_EOL_aluminium = Boiler_aluminium
Boiler_EOL_elastomer = Boiler_elastomer
Boiler_EOL_PVC = Boiler_PVC
Boiler_EOL_refrigerantgas = Boiler_refrigerantgas
Boiler_EOL_copper = Boiler_copper
Boiler_EOL_electronic = Boiler_electronic #kg
Boiler_EOL_lubricating_oil = Boiler_lubricating_oil #kg
Boiler_EOL_hazardous = Boiler_hazardous #kg

## 3.8 Module C4

In [ ]:
Boiler_EOL_landfill_stainlessSteel = Boiler_EOL_stainlessSteel * LandfillMetals2
Boiler_EOL_landfill_steel = Boiler_EOL_steel * LandfillMetals
Boiler_EOL_landfill_aluminium = Boiler_EOL_aluminium * LandfillMetals
Boiler_EOL_inc_elastomer = Boiler_EOL_elastomer * IncinerationElastomer
Boiler_EOL_landfill_elastomer = Boiler_EOL_elastomer * LandfillElastomer
Boiler_EOL_inc_PVC = Boiler_EOL_PVC * IncinerationPVC
Boiler_EOL_landfill_PVC = Boiler_EOL_PVC * LandfillPVC
Boiler_EOL_landfill_refrigerantgas = Boiler_EOL_refrigerantgas * LandfillRefrigerant
Boiler_EOL_landfill_copper = Boiler_EOL_copper * LandfillMetals
Boiler_EOL_inc_electronic = Boiler_EOL_electronic * IncinerationElectronics #kg
Boiler_EOL_landfill_electronic = Boiler_EOL_electronic * LandfillElectronics #kg
Boiler_EOL_inc_lubricatingoil = Boiler_lubricating_oil * IncinerationLubricatingOil #kg
Boiler_EOL_inc_hazardouswaste = Boiler_hazardous * IncinerationHazardous #kg


## 3.9 Module D1 (related to the export of secondary materials)

In [ ]:
Boiler_EOL_recycling_stainlessSteel = (RecyclingMetals2 * RecyclingEfficiencyMetals * Boiler_EOL_stainlessSteel) - (RecycledContentMetals * Boiler_EOL_stainlessSteel)
Boiler_EOL_primary_stainlessSteel = ((RecycledContentMetals * Boiler_EOL_stainlessSteel) - (RecyclingMetals2 * RecyclingEfficiencyMetals * Boiler_EOL_stainlessSteel)) * SRmetals

Boiler_EOL_recycling_steel = (RecyclingMetals * RecyclingEfficiencyMetals * Boiler_EOL_steel) - (RecycledContentMetals * Boiler_EOL_steel)
Boiler_EOL_primary_steel = ((RecycledContentMetals * Boiler_EOL_steel) - (RecyclingMetals * RecyclingEfficiencyMetals * Boiler_EOL_steel)) * SRmetals

Boiler_EOL_recycling_aluminium = (RecyclingMetals * RecyclingEfficiencyMetals * Boiler_EOL_aluminium) - (RecycledContentMetals * Boiler_EOL_aluminium)
Boiler_EOL_primary_aluminium = ((RecycledContentMetals * Boiler_EOL_aluminium) - (RecyclingMetals * RecyclingEfficiencyMetals * Boiler_EOL_aluminium)) * SRmetals

Boiler_EOL_recycling_copper = (RecyclingMetals * RecyclingEfficiencyMetals * Boiler_EOL_copper) - (RecycledContentMetals * Boiler_EOL_copper)
Boiler_EOL_primary_copper = ((RecycledContentMetals * Boiler_EOL_copper) - (RecyclingMetals * RecyclingEfficiencyMetals * Boiler_EOL_copper)) * SRmetals


## 3.10 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
Boiler_EOL_energyrecoveryheat_elastomer = - Boiler_EOL_inc_elastomer * LHVElastomer * Xincheat
Boiler_EOL_energyrecoveryele_elastomer = (- Boiler_EOL_inc_elastomer * LHVElastomer * Xincele) / 3.6
Boiler_EOL_energyrecoveryheat_PVC = - Boiler_EOL_inc_PVC * LHVPVC * Xincheat
Boiler_EOL_energyrecoveryele_PVC = (- Boiler_EOL_inc_PVC * LHVPVC * Xincele) / 3.6
Boiler_EOL_energyrecoveryheat_electronics = - Boiler_EOL_inc_electronic * LHVElectronics * Xincheat
Boiler_EOL_energyrecoveryele_electronics = (- Boiler_EOL_inc_electronic * LHVElectronics * Xincele) / 3.6
Boiler_EOL_energyrecoveryheat_lubricatingoil = - Boiler_EOL_inc_lubricatingoil * LHVHazardous * Xincheat
Boiler_EOL_energyrecoveryele_lubricatingoil = (- Boiler_EOL_inc_lubricatingoil * LHVHazardous * Xincele) / 3.6
Boiler_EOL_energyrecoveryheat_hazardouswaste = - Boiler_EOL_inc_hazardouswaste * LHVHazardous * Xincheat
Boiler_EOL_energyrecoveryele_hazardouswaste = (- Boiler_EOL_inc_hazardouswaste * LHVHazardous * Xincele) / 3.6

# 4. Data for export 

In [ ]:
ThermalEnergyProduced_Boiler = Energy_produced * RSP